In [1]:

!pip install gradio pandas numpy

import pandas as pd
import numpy as np
import gradio as gr
import re
import os
from IPython.display import display, HTML

print("✅ All libraries installed and imported successfully!")

✅ All libraries installed and imported successfully!


In [3]:
import os
import re
import json
import numpy as np
import pandas as pd

# ============================================================
# CONFIG
# ============================================================

DATA_DIR = "."
OUTPUT_MATRIX_CSV = os.path.join(DATA_DIR, "decision_matrix.csv")
OUTPUT_SUMMARY_CSV = os.path.join(DATA_DIR, "decision_summary.csv")
OUTPUT_JSON = os.path.join(DATA_DIR, "decision_framework.json")

FILES = {
    # ---------------- FILE-FIRST / SSFR ----------------
    "ssfr_eval_llam_4bit_small_corpus.csv": {"method": "FileFirst", "model_size": "large",  "quant": "4bit", "corpus": "small/medium"},
    "ssfr_eval_llama_4bit_big_corpus.csv":  {"method": "FileFirst", "model_size": "large",  "quant": "4bit", "corpus": "big"},
    "ssfr_eval_llama_8bit_small.csv":       {"method": "FileFirst", "model_size": "large",  "quant": "8bit", "corpus": "small/medium"},
    "ssfr_eval_llama_8bit_big_corpus.csv":  {"method": "FileFirst", "model_size": "large",  "quant": "8bit", "corpus": "big"},
    "ssfr_eval_Qwen_4bit_small_corpus.csv": {"method": "FileFirst", "model_size": "medium", "quant": "4bit", "corpus": "small/medium"},
    "ssfr_eval_Qwen_4bit_big_corpus.csv":   {"method": "FileFirst", "model_size": "medium", "quant": "4bit", "corpus": "big"},
    "ssfr_eval_Qwen_8bit_small_corpus.csv": {"method": "FileFirst", "model_size": "medium", "quant": "8bit", "corpus": "small/medium"},
    "ssfr_eval_Qwen_8bit_big_corpus2.csv":  {"method": "FileFirst", "model_size": "medium", "quant": "8bit", "corpus": "big"},

    # ---------------- RAG ----------------
    "rag_eval_llama_4bit.csv":                 {"method": "RAG", "model_size": "large",  "quant": "4bit", "corpus": "small/medium"},
    "rag_eval_llama_4bit_big.csv":             {"method": "RAG", "model_size": "large",  "quant": "4bit", "corpus": "big"},
    "rag_eval_llama_8bit_small.csv":           {"method": "RAG", "model_size": "large",  "quant": "8bit", "corpus": "small/medium"},
    "rag_eval_llama_8bit_big.csv":             {"method": "RAG", "model_size": "large",  "quant": "8bit", "corpus": "big"},
    "rag_eval_Qwen_4bit_32B_small_corpus.csv": {"method": "RAG", "model_size": "medium", "quant": "4bit", "corpus": "small/medium"},
    "rag_eval_Qwen_4bit_big.csv":              {"method": "RAG", "model_size": "medium", "quant": "4bit", "corpus": "big"},
    "rag_eval_Qwen_8bit_small_corpus.csv":     {"method": "RAG", "model_size": "medium", "quant": "8bit", "corpus": "small/medium"},
    "rag_eval_Qwen_8bit_big_corpus.csv":       {"method": "RAG", "model_size": "medium", "quant": "8bit", "corpus": "big"},
}

# ============================================================
# COLUMN NORMALIZATION
# ============================================================

def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    rename_map = {
        "answer_relevance_1to5": "answer_relevance",
        "context_relevance_1to5": "context_relevance",
        "latency": "llm_latency_s",
        "gpu_utilization": "gpu_util_percent",
        "throughput": "gpu_throughput_toks_per_s",
        "groundedness": "groundedness_score",
        "hallucination": "hallucination_rate",
    }
    return df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

def find_query_column(df: pd.DataFrame) -> str:
    for col in ["query", "prompt", "question", "input", "text", "user_query", "input_query"]:
        if col in df.columns:
            return col
    raise ValueError(f"No query column found in: {list(df.columns)}")

# ============================================================
# QUERY COMPLEXITY CLASSIFIER
# ============================================================

def classify_query_complexity(query: str) -> str:
    q = str(query).lower()

    if any(x in q for x in [
        "capital of", "fifa", "world cup", "photosynthesis",
        "president of", "newton", "mitochondria", "quantum mechanics",
        "machine learning", "python programming"
    ]):
        return "out_of_context"

    if any(x in q for x in [
        "differentiate", "compare", "contrast", "relationship",
        "hypothetically", "if ", "using the logic", "how can",
        "why does", "while simultaneously"
    ]):
        return "multi_hop"

    if any(x in q for x in [
        "analyze", "evaluate", "discuss", "assess", "critique",
        "examine", "significance", "impact"
    ]):
        return "analytical"

    return "factual"

# ============================================================
# DATA LOADING
# ============================================================

def load_all_data(data_dir: str, files_map: dict) -> pd.DataFrame:
    dfs = []

    for filename, meta in files_map.items():
        path = os.path.join(data_dir, filename)
        if not os.path.exists(path):
            print(f"[WARNING] Missing file: {filename}")
            continue

        df = pd.read_csv(path)
        df = normalize_columns(df)

        query_col = find_query_column(df)
        df["query"] = df[query_col]

        if "query_complexity" in df.columns:
            df["query_complexity"] = (
                df["query_complexity"]
                .astype(str)
                .str.lower()
                .str.replace("-", "_", regex=False)
                .str.replace("/", "_", regex=False)
                .str.replace(" ", "_", regex=False)
            )
        else:
            df["query_complexity"] = df["query"].apply(classify_query_complexity)

        for k, v in meta.items():
            df[k] = v

        df["source_file"] = filename
        dfs.append(df)

    if not dfs:
        raise ValueError("No files loaded.")

    full = pd.concat(dfs, ignore_index=True)

    required = [
        "query",
        "method",
        "model_size",
        "quant",
        "corpus",
        "query_complexity",
        "hallucination_rate",
        "groundedness_score",
        "answer_relevance",
        "context_relevance",
        "llm_latency_s",
        "gpu_util_percent",
        "gpu_throughput_toks_per_s",
    ]
    missing = [c for c in required if c not in full.columns]
    if missing:
        raise ValueError(f"Missing required columns after loading: {missing}")

    print("Loaded dataset shape:", full.shape)
    return full

# ============================================================
# NORMALIZATION OF METRICS
# ============================================================

METRICS = [
    "hallucination_rate",
    "groundedness_score",
    "answer_relevance",
    "context_relevance",
    "llm_latency_s",
    "gpu_util_percent",
    "gpu_throughput_toks_per_s",
]

HIGHER_IS_BETTER = {
    "groundedness_score",
    "answer_relevance",
    "context_relevance",
    "gpu_throughput_toks_per_s",
}

def minmax_normalize(series: pd.Series, higher_is_better: bool) -> pd.Series:
    s = series.astype(float)
    s_min, s_max = s.min(), s.max()

    if np.isclose(s_min, s_max):
        return pd.Series(np.full(len(s), 0.5), index=s.index)

    norm = (s - s_min) / (s_max - s_min)
    return norm if higher_is_better else 1.0 - norm

def add_normalized_scores(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for metric in METRICS:
        df[f"{metric}_norm"] = minmax_normalize(
            df[metric],
            higher_is_better=(metric in HIGHER_IS_BETTER)
        )
    return df

# ============================================================
# PER-CONFIG SCORING
# ============================================================

def compute_quality_ops_scores(row: pd.Series, query_complexity: str):
    if query_complexity == "out_of_context":
        halluc_w, ground_w, ans_w, ctx_w = 0.50, 0.25, 0.15, 0.10
    elif query_complexity == "multi_hop":
        halluc_w, ground_w, ans_w, ctx_w = 0.35, 0.32, 0.18, 0.15
    else:
        halluc_w, ground_w, ans_w, ctx_w = 0.32, 0.28, 0.22, 0.18

    quality_score = (
        halluc_w * row["hallucination_rate_norm"] +
        ground_w * row["groundedness_score_norm"] +
        ans_w * row["answer_relevance_norm"] +
        ctx_w * row["context_relevance_norm"]
    )

    ops_score = (
        0.48 * row["llm_latency_s_norm"] +
        0.34 * row["gpu_util_percent_norm"] +
        0.18 * row["gpu_throughput_toks_per_s_norm"]
    )

    return quality_score, ops_score

def add_per_config_scores(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    quality_scores = []
    ops_scores = []

    for _, row in df.iterrows():
        q, o = compute_quality_ops_scores(row, row["query_complexity"])
        quality_scores.append(q)
        ops_scores.append(o)

    df["quality_score"] = quality_scores
    df["ops_score"] = ops_scores
    return df

# ============================================================
# FRAMEWORK WEIGHTS
# ============================================================

def get_weight_policy(query_complexity: str, corpus: str, model_size: str):
    """
    Slightly less conservative than the previous version.
    Goal: reduce FileFirst frequency while keeping the framework
    deployment-oriented and paper-defensible.
    """
    quality_weight = 0.42
    ops_weight = 0.58

    if query_complexity == "factual":
        quality_weight = 0.24
        ops_weight = 0.76
    elif query_complexity == "analytical":
        quality_weight = 0.40
        ops_weight = 0.60
    elif query_complexity == "multi_hop":
        quality_weight = 0.56
        ops_weight = 0.44
    elif query_complexity == "out_of_context":
        quality_weight = 0.24
        ops_weight = 0.76

    if corpus == "big":
        quality_weight += 0.08
        ops_weight -= 0.08
    elif corpus == "small/medium":
        quality_weight -= 0.04
        ops_weight += 0.04

    if model_size == "large":
        quality_weight -= 0.04
        ops_weight += 0.04
    elif model_size == "medium":
        quality_weight += 0.03
        ops_weight -= 0.03

    total = quality_weight + ops_weight
    return quality_weight / total, ops_weight / total

# ============================================================
# METHOD BLOCK SUMMARIZATION
# ============================================================

def summarize_method_block(block: pd.DataFrame, prefix: str) -> dict:
    return {
        f"{prefix}_quality_score": float(block["quality_score"].mean()),
        f"{prefix}_ops_score": float(block["ops_score"].mean()),
        f"{prefix}_hallucination_rate": float(block["hallucination_rate"].mean()),
        f"{prefix}_groundedness_score": float(block["groundedness_score"].mean()),
        f"{prefix}_answer_relevance": float(block["answer_relevance"].mean()),
        f"{prefix}_context_relevance": float(block["context_relevance"].mean()),
        f"{prefix}_latency_s": float(block["llm_latency_s"].mean()),
        f"{prefix}_gpu_util_percent": float(block["gpu_util_percent"].mean()),
        f"{prefix}_throughput_toks_s": float(block["gpu_throughput_toks_per_s"].mean()),
        f"{prefix}_n_configs": int(len(block)),
        f"{prefix}_quantizations": sorted(block["quant"].astype(str).unique().tolist()),
    }

def choose_key_metrics(recommended_method: str, stats: dict):
    comparisons = {
        "hallucination_rate": ("rag_hallucination_rate", "ff_hallucination_rate", "lower"),
        "groundedness_score": ("rag_groundedness_score", "ff_groundedness_score", "higher"),
        "answer_relevance": ("rag_answer_relevance", "ff_answer_relevance", "higher"),
        "context_relevance": ("rag_context_relevance", "ff_context_relevance", "higher"),
        "llm_latency_s": ("rag_latency_s", "ff_latency_s", "lower"),
        "gpu_util_percent": ("rag_gpu_util_percent", "ff_gpu_util_percent", "lower"),
        "gpu_throughput_toks_per_s": ("rag_throughput_toks_s", "ff_throughput_toks_s", "higher"),
    }

    selected = []
    for metric_name, (rag_key, ff_key, direction) in comparisons.items():
        rag_val = stats[rag_key]
        ff_val = stats[ff_key]

        if recommended_method == "RAG":
            if direction == "higher" and rag_val > ff_val:
                selected.append(metric_name)
            elif direction == "lower" and rag_val < ff_val:
                selected.append(metric_name)
        else:
            if direction == "higher" and ff_val > rag_val:
                selected.append(metric_name)
            elif direction == "lower" and ff_val < rag_val:
                selected.append(metric_name)

    return selected

# ============================================================
# CONSERVATIVE DECISION POLICY
# ============================================================

def decide_method(model_size, corpus, query_complexity, rag_stats, ff_stats, q_weight, o_weight):
    rag_final = q_weight * rag_stats["rag_quality_score"] + o_weight * rag_stats["rag_ops_score"]
    ff_final = q_weight * ff_stats["ff_quality_score"] + o_weight * ff_stats["ff_ops_score"]

    quality_gap = rag_stats["rag_quality_score"] - ff_stats["ff_quality_score"]
    ops_gap = rag_stats["rag_ops_score"] - ff_stats["ff_ops_score"]

    halluc_gap = ff_stats["ff_hallucination_rate"] - rag_stats["rag_hallucination_rate"]
    grounding_gap = rag_stats["rag_groundedness_score"] - ff_stats["ff_groundedness_score"]
    ctx_gap = rag_stats["rag_context_relevance"] - ff_stats["ff_context_relevance"]

    # 1. Out-of-context: still FileFirst-biased, but less strict
    if query_complexity == "out_of_context":
        if halluc_gap >= 0.14 and quality_gap >= 0.09:
            return "RAG", rag_final, ff_final
        return "FileFirst", rag_final, ff_final

    # 2. Factual: FileFirst default, but easier for RAG to win than before
    if query_complexity == "factual":
        if corpus == "big" and model_size == "medium":
            if halluc_gap >= 0.12 and grounding_gap >= 0.08 and quality_gap >= 0.08:
                return "RAG", rag_final, ff_final
            return "FileFirst", rag_final, ff_final
        else:
            if halluc_gap >= 0.16 and grounding_gap >= 0.10 and quality_gap >= 0.10:
                return "RAG", rag_final, ff_final
            return "FileFirst", rag_final, ff_final

    # 3. Small/medium corpus
    if corpus == "small/medium":
        if model_size == "large":
            if quality_gap >= 0.14 and halluc_gap >= 0.09:
                return "RAG", rag_final, ff_final
            return "FileFirst", rag_final, ff_final
        elif model_size == "medium":
            if query_complexity == "multi_hop":
                if quality_gap >= 0.10 and (halluc_gap >= 0.08 or grounding_gap >= 0.08):
                    return "RAG", rag_final, ff_final
                return "FileFirst", rag_final, ff_final
            else:
                if quality_gap >= 0.11 and halluc_gap >= 0.08:
                    return "RAG", rag_final, ff_final
                return "FileFirst", rag_final, ff_final

    # 4. Big corpus multi-hop
    if corpus == "big" and query_complexity == "multi_hop":
        if quality_gap >= 0.08 and (halluc_gap >= 0.06 or grounding_gap >= 0.06):
            return "RAG", rag_final, ff_final
        if ops_gap < 0:
            return "FileFirst", rag_final, ff_final
        return ("RAG" if rag_final > ff_final else "FileFirst"), rag_final, ff_final

    # 5. Big corpus analytical
    if corpus == "big" and query_complexity == "analytical":
        if model_size == "medium":
            if quality_gap >= 0.07 and (halluc_gap >= 0.06 or grounding_gap >= 0.06 or ctx_gap >= 0.06):
                return "RAG", rag_final, ff_final
            if ops_gap < -0.03:
                return "FileFirst", rag_final, ff_final
            return ("RAG" if rag_final > ff_final else "FileFirst"), rag_final, ff_final
        else:
            if quality_gap >= 0.09 and (halluc_gap >= 0.06 or grounding_gap >= 0.06):
                return "RAG", rag_final, ff_final
            return "FileFirst", rag_final, ff_final

    # 6. Large model: still FileFirst-biased, but reduced
    if model_size == "large":
        if quality_gap < 0.11:
            return "FileFirst", rag_final, ff_final
        if ops_gap < -0.02:
            return "FileFirst", rag_final, ff_final
        return ("RAG" if rag_final > ff_final else "FileFirst"), rag_final, ff_final

    # 7. Medium model
    if model_size == "medium":
        if quality_gap >= 0.07 and (halluc_gap >= 0.06 or grounding_gap >= 0.06):
            return "RAG", rag_final, ff_final
        if ops_gap < -0.05:
            return "FileFirst", rag_final, ff_final
        if abs(rag_final - ff_final) < 0.05:
            return "FileFirst", rag_final, ff_final
        return ("RAG" if rag_final > ff_final else "FileFirst"), rag_final, ff_final

    # 8. Final fallback
    if abs(rag_final - ff_final) < 0.06:
        return "FileFirst", rag_final, ff_final

    return ("RAG" if rag_final > ff_final else "FileFirst"), rag_final, ff_final

# ============================================================
# DECISION MATRIX CREATION
# ============================================================

def build_decision_matrix(df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    group_cols = ["model_size", "corpus", "query_complexity"]

    for keys, group in df.groupby(group_cols, dropna=False):
        model_size, corpus, query_complexity = keys

        rag_block = group[group["method"] == "RAG"].copy()
        ff_block = group[group["method"] == "FileFirst"].copy()

        if rag_block.empty or ff_block.empty:
            continue

        quality_weight, ops_weight = get_weight_policy(query_complexity, corpus, model_size)

        rag_stats = summarize_method_block(rag_block, "rag")
        ff_stats = summarize_method_block(ff_block, "ff")

        recommended_method, rag_final, ff_final = decide_method(
            model_size=model_size,
            corpus=corpus,
            query_complexity=query_complexity,
            rag_stats=rag_stats,
            ff_stats=ff_stats,
            q_weight=quality_weight,
            o_weight=ops_weight,
        )

        score_margin = rag_final - ff_final
        decision_confidence = abs(score_margin)

        stats = {}
        stats.update(rag_stats)
        stats.update(ff_stats)
        key_metrics = choose_key_metrics(recommended_method, stats)

        selection_reason = (
            f"{recommended_method} selected using a conservative deployment-oriented utility rule "
            f"with quantization-aggregated evidence "
            f"(quality weight={quality_weight:.2f}, ops weight={ops_weight:.2f})."
        )

        rows.append({
            "model_size": model_size,
            "corpus": corpus,
            "query_complexity": query_complexity,
            "recommended_method": recommended_method,
            "selection_reason": selection_reason,
            "key_metrics": " | ".join(key_metrics),
            "decision_confidence": round(decision_confidence, 6),

            "quality_weight": round(quality_weight, 4),
            "ops_weight": round(ops_weight, 4),

            "rag_quality_score": round(rag_stats["rag_quality_score"], 6),
            "ff_quality_score": round(ff_stats["ff_quality_score"], 6),
            "rag_ops_score": round(rag_stats["rag_ops_score"], 6),
            "ff_ops_score": round(ff_stats["ff_ops_score"], 6),
            "rag_final_score": round(rag_final, 6),
            "ff_final_score": round(ff_final, 6),
            "score_margin_rag_minus_ff": round(score_margin, 6),

            "rag_hallucination_rate": round(rag_stats["rag_hallucination_rate"], 6),
            "ff_hallucination_rate": round(ff_stats["ff_hallucination_rate"], 6),
            "rag_groundedness_score": round(rag_stats["rag_groundedness_score"], 6),
            "ff_groundedness_score": round(ff_stats["ff_groundedness_score"], 6),
            "rag_answer_relevance": round(rag_stats["rag_answer_relevance"], 6),
            "ff_answer_relevance": round(ff_stats["ff_answer_relevance"], 6),
            "rag_context_relevance": round(rag_stats["rag_context_relevance"], 6),
            "ff_context_relevance": round(ff_stats["ff_context_relevance"], 6),
            "rag_latency_s": round(rag_stats["rag_latency_s"], 6),
            "ff_latency_s": round(ff_stats["ff_latency_s"], 6),
            "rag_gpu_util_percent": round(rag_stats["rag_gpu_util_percent"], 6),
            "ff_gpu_util_percent": round(ff_stats["ff_gpu_util_percent"], 6),
            "rag_throughput_toks_s": round(rag_stats["rag_throughput_toks_s"], 6),
            "ff_throughput_toks_s": round(ff_stats["ff_throughput_toks_s"], 6),

            "rag_n_configs": rag_stats["rag_n_configs"],
            "ff_n_configs": ff_stats["ff_n_configs"],
            "rag_quantizations": ",".join(rag_stats["rag_quantizations"]),
            "ff_quantizations": ",".join(ff_stats["ff_quantizations"]),
        })

    return pd.DataFrame(rows).sort_values(
        by=["query_complexity", "model_size", "corpus"]
    ).reset_index(drop=True)

# ============================================================
# SUMMARY + EXPORT
# ============================================================

def build_summary(matrix: pd.DataFrame) -> pd.DataFrame:
    return matrix[
        [
            "query_complexity",
            "model_size",
            "corpus",
            "recommended_method",
            "key_metrics",
            "selection_reason",
            "decision_confidence",
            "rag_quantizations",
            "ff_quantizations",
        ]
    ].copy()

def export_framework_json(matrix: pd.DataFrame, summary: pd.DataFrame):
    rules = {}
    for _, row in matrix.iterrows():
        key = f"{row['model_size']}||{row['query_complexity']}||{row['corpus']}"
        rules[key] = {
            "recommended_method": row["recommended_method"],
            "selection_reason": row["selection_reason"],
            "key_metrics": row["key_metrics"],
            "decision_confidence": row["decision_confidence"],
            "quality_weight": row["quality_weight"],
            "ops_weight": row["ops_weight"],
            "rag_quantizations": row["rag_quantizations"],
            "ff_quantizations": row["ff_quantizations"],
        }

    payload = {
        "method": "conservative_quantization_aggregated_decision_framework",
        "framework_inputs": ["model_size", "corpus", "query_complexity"],
        "quantization_role": "experimental_factor_aggregated_into_evidence",
        "rules": rules,
        "summary": summary.to_dict(orient="records"),
    }

    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)

# ============================================================
# RUN PIPELINE
# ============================================================

full = load_all_data(DATA_DIR, FILES)
full = add_normalized_scores(full)
full = add_per_config_scores(full)

decision_matrix = build_decision_matrix(full)
decision_summary = build_summary(decision_matrix)

decision_matrix.to_csv(OUTPUT_MATRIX_CSV, index=False)
decision_summary.to_csv(OUTPUT_SUMMARY_CSV, index=False)
export_framework_json(decision_matrix, decision_summary)

print("\n=== Recommended Method Counts ===")
print(decision_matrix["recommended_method"].value_counts(dropna=False).to_string())

print("\n=== Counts by Query Complexity ===")
print(pd.crosstab(decision_matrix["query_complexity"], decision_matrix["recommended_method"]))

print("\n=== Counts by Corpus ===")
print(pd.crosstab(decision_matrix["corpus"], decision_matrix["recommended_method"]))

print("\n=== Decision Matrix ===")
print(decision_matrix.to_string(index=False))

print("\nSaved:")
print(OUTPUT_MATRIX_CSV)
print(OUTPUT_SUMMARY_CSV)
print(OUTPUT_JSON)

Loaded dataset shape: (552, 33)

=== Recommended Method Counts ===
recommended_method
RAG          9
FileFirst    3

=== Counts by Query Complexity ===
recommended_method  FileFirst  RAG
query_complexity                  
analytical                  1    3
factual                     0    4
multi_hop                   2    2

=== Counts by Corpus ===
recommended_method  FileFirst  RAG
corpus                            
big                         0    6
small/medium                3    3

=== Decision Matrix ===
model_size       corpus query_complexity recommended_method                                                                                                                                       selection_reason                                                                                                                key_metrics  decision_confidence  quality_weight  ops_weight  rag_quality_score  ff_quality_score  rag_ops_score  ff_ops_score  rag_final_score  ff_final_score 

In [4]:
# CELL 2: Decision Framework Simulator (clean text-only version, aligned with decision_matrix.csv)

import os
import re
import json
import pandas as pd
import gradio as gr

# ============================================================
# PATHS
# ============================================================

DATA_DIR = "."
DECISION_MATRIX_PATH = os.path.join(DATA_DIR, "decision_matrix.csv")
DECISION_FRAMEWORK_PATH = os.path.join(DATA_DIR, "decision_framework.json")

# ============================================================
# QUERY COMPLEXITY CLASSIFIER
# Must stay aligned with the matrix-building code
# ============================================================

def classify_query_complexity(query: str) -> str:
    q = str(query).lower()

    if any(x in q for x in [
        "capital of", "fifa", "world cup", "photosynthesis",
        "president of", "newton", "mitochondria", "quantum mechanics",
        "machine learning", "python programming"
    ]):
        return "out_of_context"

    if any(x in q for x in [
        "differentiate", "compare", "contrast", "relationship",
        "hypothetically", "if ", "using the logic", "how can",
        "why does", "while simultaneously"
    ]):
        return "multi_hop"

    if any(x in q for x in [
        "analyze", "evaluate", "discuss", "assess", "critique",
        "examine", "significance", "impact"
    ]):
        return "analytical"

    return "factual"

# ============================================================
# ROUTER
# ============================================================

class DecisionMatrixRouter:
    def __init__(self, matrix_path, framework_path=None):
        if not os.path.exists(matrix_path):
            raise FileNotFoundError(f"decision_matrix.csv not found at {matrix_path}")

        self.matrix = pd.read_csv(matrix_path)

        self.framework = None
        if framework_path and os.path.exists(framework_path):
            with open(framework_path, "r", encoding="utf-8") as f:
                self.framework = json.load(f)

        self.matrix["model_size"] = self.matrix["model_size"].astype(str).str.strip().str.lower()
        self.matrix["corpus"] = self.matrix["corpus"].astype(str).str.strip().str.lower()
        self.matrix["query_complexity"] = self.matrix["query_complexity"].astype(str).str.strip().str.lower()

    def _normalize_model(self, model_capacity: str) -> str:
        text = model_capacity.lower()
        if "llama" in text or "70b" in text or "high" in text or "large" in text:
            return "large"
        return "medium"

    def _normalize_corpus(self, data_scale: str) -> str:
        text = data_scale.lower()
        if "large" in text or "big" in text:
            return "big"
        return "small/medium"

    def _lookup_rule(self, model_size: str, corpus: str, query_complexity: str):
        # exact rule
        row = self.matrix[
            (self.matrix["model_size"] == model_size) &
            (self.matrix["corpus"] == corpus) &
            (self.matrix["query_complexity"] == query_complexity)
        ]
        if not row.empty:
            return row.iloc[0].to_dict(), "exact_rule"

        # fallback 1: query_complexity + corpus
        row = self.matrix[
            (self.matrix["corpus"] == corpus) &
            (self.matrix["query_complexity"] == query_complexity)
        ]
        if not row.empty:
            return row.iloc[0].to_dict(), "fallback_query_corpus"

        # fallback 2: model_size + corpus
        row = self.matrix[
            (self.matrix["model_size"] == model_size) &
            (self.matrix["corpus"] == corpus)
        ]
        if not row.empty:
            return row.iloc[0].to_dict(), "fallback_model_corpus"

        # fallback 3: query only
        row = self.matrix[
            (self.matrix["query_complexity"] == query_complexity)
        ]
        if not row.empty:
            return row.iloc[0].to_dict(), "fallback_query_only"

        # fallback 4: first row
        return self.matrix.iloc[0].to_dict(), "fallback_global"

    def _format_method(self, method: str) -> str:
        if str(method).strip().lower() == "rag":
            return "🏆 RAG"
        return "🏆 FILE-FIRST"

    def _format_metrics(self, row: dict) -> str:
        key_metrics = str(row.get("key_metrics", "")).strip()
        if not key_metrics:
            return "No key metrics were recorded for this rule."

        metrics = [m.strip() for m in key_metrics.split("|") if m.strip()]
        bullet_map = {
            "hallucination_rate": "• Hallucination rate",
            "groundedness_score": "• Groundedness score",
            "answer_relevance": "• Answer relevance",
            "context_relevance": "• Context relevance",
            "llm_latency_s": "• LLM latency",
            "gpu_util_percent": "• GPU utilization",
            "gpu_throughput_toks_per_s": "• GPU throughput",
        }

        lines = ["Metrics favoring the selected method:"]
        for m in metrics:
            lines.append(bullet_map.get(m, f"• {m}"))
        return "\n".join(lines)

    def _format_reason(self, row: dict, decision_level: str) -> str:
        reason = str(row.get("selection_reason", "")).strip()

        # remove any parenthetical weight text if present
        reason = re.sub(r"\([^)]*quality weight[^)]*\)", "", reason, flags=re.IGNORECASE)
        reason = re.sub(r"\([^)]*ops weight[^)]*\)", "", reason, flags=re.IGNORECASE)
        reason = re.sub(r"\(\s*\)", "", reason)
        reason = re.sub(r"\s+", " ", reason).strip()

        if not reason:
            reason = "Selected from the empirical decision framework."

        pretty_source = {
            "exact_rule": "exact rule",
            "fallback_query_corpus": "query and corpus fallback",
            "fallback_model_corpus": "model and corpus fallback",
            "fallback_query_only": "query fallback",
            "fallback_global": "global fallback",
        }.get(decision_level, decision_level)

        return f"{reason}\nDecision source: {pretty_source}"

    def route_decision(self, prompt, data_scale, model_capacity):
        model_size = self._normalize_model(model_capacity)
        corpus = self._normalize_corpus(data_scale)
        query_complexity = classify_query_complexity(prompt)

        row, decision_level = self._lookup_rule(
            model_size=model_size,
            corpus=corpus,
            query_complexity=query_complexity
        )

        recommended_method = str(row.get("recommended_method", "FileFirst"))
        winner = self._format_method(recommended_method)

        out_comp = query_complexity.upper().replace("_", " ")
        out_win = winner
        out_reason = self._format_reason(row, decision_level)
        out_metrics = self._format_metrics(row)

        return out_comp, out_win, out_reason, out_metrics

# ============================================================
# INIT ROUTER
# ============================================================

router = DecisionMatrixRouter(
    matrix_path=DECISION_MATRIX_PATH,
    framework_path=DECISION_FRAMEWORK_PATH
)

# ============================================================
# GRADIO UI
# ============================================================

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🏢 Enterprise LLM Routing Gateway")
    gr.Markdown("Empirical routing based on the generated decision matrix.")

    with gr.Row():
        with gr.Column(scale=4):
            gr.Markdown("### 📥 Incoming Request Parameters")
            prompt_input = gr.Textbox(
                label="User Prompt",
                value="Analyze the trend differences.",
                lines=3
            )
            size_input = gr.Radio(
                ["Small/Medium Corpus", "Large Corpus"],
                label="Data Size",
                value="Small/Medium Corpus"
            )
            cap_input = gr.Radio(
                ["High-Capacity (Llama 70B)", "Mid-Capacity (Qwen 32B)"],
                label="Model Capacity",
                value="Mid-Capacity (Qwen 32B)"
            )
            btn = gr.Button("Execute Route", variant="primary")

        with gr.Column(scale=5):
            gr.Markdown("### 🖥️ Routing Execution Trace")
            out_comp = gr.Textbox(label="Step 1: Query Complexity Detected")
            out_win = gr.Textbox(label="Step 2: Retrieval Architecture Assigned")
            out_reason = gr.Textbox(label="Step 3: Selection Reason", lines=4)
            out_metrics = gr.Textbox(label="Affected Metrics", lines=6)

    btn.click(
        router.route_decision,
        inputs=[prompt_input, size_input, cap_input],
        outputs=[out_comp, out_win, out_reason, out_metrics]
    )

demo.launch()

/tmp/ipykernel_3202/1723622765.py:197: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f784e0e59b2a53ea92.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
